# RF Research

In [1]:
import pandas as pd
import ta
import optuna
import sklearn
from sklearn.ensemble import RandomForestClassifier
pd.options.plotting.backend = "plotly"

from datetime import datetime, timedelta
import pytz

import numpy as np
np.random.seed(42)

optuna.logging.set_verbosity(optuna.logging.CRITICAL)

/Users/anzekravanja/Projects/smart-trader/conda-env/lib/python3.8/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Get the data

In [ ]:
def get_candlestick_data(symbol, start_date, end_date):
    # The TD API expects a timestamp in milliseconds. However, the timestamp() 
    # method only returns to seconds so multiply it by 1000.
    today_unix = str(int(round(end_date.timestamp() * 1000)))
    today_ago_unix = str(int(round(start_date.timestamp() * 1000)))

    # Make the request
    cs = TDSession.get_price_history(
        symbol=symbol,
        period_type='month',
        frequency_type='daily',
        start_date=today_ago_unix,
        end_date=today_unix,
        extended_hours=False
    )
    df = pd.DataFrame.from_records(cs['candles'])
    df['ts'] = pd.to_datetime(df['datetime'], unit='ms', utc=True)
    df.sort_values(by=['datetime'])
    
    return df

df = get_candlestick_data(SYMBOL, datetime.utcnow() - timedelta(days=28*12*3), datetime.utcnow())

In [2]:
df = pd.read_csv('../data/BTCUSD.csv', parse_dates=True)
df['ts'] = df['datetime']
df = df.iloc[-28*12*3:]

In [3]:
df.head()

,close,high,low,open,volume,datetime,ts
3146,5870.90,6278.24,5869.00,6275.11,9298.939051,2020-03-29 00:00:00+00:00,2020-03-29 00:00:00+00:00
3147,6407.77,6630.00,5856.00,5878.47,14974.444163,2020-03-30 00:00:00+00:00,2020-03-30 00:00:00+00:00
3148,6421.14,6527.24,6337.42,6408.95,6342.298327,2020-03-31 00:00:00+00:00,2020-03-31 00:00:00+00:00
3149,6652.07,6710.21,6137.71,6428.74,11319.147938,2020-04-01 00:00:00+00:00,2020-04-01 00:00:00+00:00
3150,6801.99,7236.39,6575.13,6669.95,17212.820673,2020-04-02 00:00:00+00:00,2020-04-02 00:00:00+00:00


In [4]:
def create_dataset(df, lag=0, window=14, window_slow=22, window_fast=12, mode='train', **kwargs):
    df_t = df.copy()

    # BollingerBands
    bb = ta.volatility.BollingerBands(close=df.close, window=window, window_dev=2)
    df_t['bb_bbh'] = 1 - bb.bollinger_hband() / df.close
    df_t['bb_bbl'] = 1 - bb.bollinger_lband() / df.close
    df_t['bb_bbw'] = bb.bollinger_wband() / df.close
    df_t['bb_bbp'] = bb.bollinger_pband()

    # RSI
    rsi = ta.momentum.RSIIndicator(close=df.close, window=window)
    srsi = ta.momentum.StochRSIIndicator(close=df.close, window=window)
    df_t['rsi'] = rsi.rsi()
    df_t['srsi'] = srsi.stochrsi()

    # MACD
    macd = ta.trend.MACD(close=df.close, window_slow=window_slow, window_fast=window_fast)
    df_t['macd'] = macd.macd() / macd.macd().std()
    df_t['macd_diff'] = macd.macd_diff() / macd.macd_diff().std()
    df_t['macd_signal'] = macd.macd_signal() / macd.macd_signal().std()

    # ATR
    atr = ta.volatility.AverageTrueRange(high=df.high, low=df.low, close=df.close, window=window)
    df_t['atr'] = 100 * atr.average_true_range() / df.close

    # StochasticOscillator
    so = ta.momentum.StochasticOscillator(high=df.high, low=df.low, close=df.close, window=window)
    df_t['so'] = so.stoch()
    df_t['sos'] = so.stoch_signal()

    # PSARIndicator
    sar = ta.trend.PSARIndicator(high=df.high, low=df.low, close=df.close)
    df_t['sar'] = sar.psar()

    # MFIIndicator
    mfi = ta.volume.MFIIndicator(high=df.high, low=df.low, close=df.close, volume=df.volume, window=window)
    df_t['mfi'] = mfi.money_flow_index()

    # VolumePriceTrendIndicator
    vpt = ta.volume.VolumePriceTrendIndicator(close=df.close, volume=df.volume)
    df_t['vpt'] = vpt.volume_price_trend()

    # TRIX
    trix = ta.trend.TRIXIndicator(close=df.close, window=window)
    df_t['trix'] = trix.trix()

    df_t = df_t.drop(['ts', 'open', 'high', 'low', 'close', 'volume', 'datetime'], axis=1)
    for c in df_t.columns:
        for l in range(lag):
            df_t[f'{c}_lag_{l + 1}'] = df_t[c].shift(l + 1)
            
    # candles
#     df_t['cdl_body'] = df.close - df.open
#     df_t['cdl_up_tck'] = df.high - df[['close', 'open']].max(axis=1)
#     df_t['cdl_dn_tck'] = df[['close', 'open']].min(axis=1) - df.low
    
#     for i in [3]:
#         df_t[f'cdl_pos_l{i}'] = df_t['cdl_body'].rolling(i).apply(lambda x: (x > 0).sum())
#         df_t[f'cdl_b_l{i}'] = df_t['cdl_body'].rolling(i).apply(lambda x: x[(x > 0)].sum()) / df_t['cdl_body'].rolling(i).apply(lambda x: np.abs(x).sum())
                                                                                                                            
#         df_t[f'cdl_bap_l{i}'] = 100 * df_t['cdl_body'].rolling(i).apply(lambda x: x[(x > 0)].mean()) / df.close
#         df_t[f'cdl_ban_l{i}'] = 100 *  df_t['cdl_body'].rolling(i).apply(lambda x: abs(x[(x < 0)].mean())) / df.close
        
#         df_t[f'cdl_btp_l{i}'] = 100 * df_t['cdl_up_tck'].rolling(i).apply(lambda x: x.mean()) / df.close
#         df_t[f'cdl_btn_l{i}'] = 100 *  df_t['cdl_dn_tck'].rolling(i).apply(lambda x: x.mean()) / df.close
    
#     df_t = df_t.drop(['cdl_body', 'cdl_up_tck', 'cdl_dn_tck'], axis=1)

    if mode == 'train':
        df_t['target'] = (df['close'].shift(-5) - df['close']).map(
            lambda c: None if np.isnan(c) else 1 if c >= 0 else 0
        )
        df_t = df_t.dropna()
        return df_t.drop('target', axis=1), df_t['target']

    return df_t, None

In [5]:
X, y = create_dataset(df)
X.head(15)

,bb_bbh,bb_bbl,bb_bbw,bb_bbp,rsi,srsi,macd,macd_diff,macd_signal,atr,so,sos,sar,mfi,vpt,trix
3186,-0.036595,0.260939,0.003419,0.877007,78.809092,0.698413,0.444719,0.288957,0.379036,4.828651,89.677041,91.144552,8199.828400,90.941512,1929.225232,1.083998
3187,-0.069634,0.216674,0.003237,0.756787,72.242495,0.237580,0.440090,0.218622,0.396050,4.901653,79.627655,88.848668,8387.245560,84.861375,-674.669785,1.137860
3188,-0.160593,0.117745,0.003121,0.423029,56.125272,0.000000,0.385689,0.027463,0.398187,6.167893,44.897557,71.400751,10074.000000,72.175122,-2995.576873,1.148370
3189,-0.171067,0.077189,0.002766,0.310925,53.610590,0.000000,0.329728,-0.129694,0.388094,6.668613,38.035479,54.186897,10034.700000,61.509589,-3330.766635,1.122981
3190,-0.120211,0.069352,0.002095,0.365852,56.920442,0.107638,0.296052,-0.195022,0.372916,6.378557,47.025279,43.319438,9996.186000,65.193293,-214.088210,1.081215
3191,-0.065006,0.113978,0.001970,0.636807,62.492105,0.288830,0.293081,-0.164069,0.360148,6.082160,61.399491,48.820083,9958.442280,63.432406,1122.960627,1.043354
3192,-0.024788,0.151975,0.001927,0.859765,66.978445,0.434727,0.313208,-0.076707,0.354178,5.875674,85.725191,64.716654,8109.000000,63.083488,1731.638601,1.021403
3193,-0.080075,0.101925,0.001977,0.560028,59.204072,0.181902,0.297713,-0.103360,0.346134,6.304011,60.926718,69.350466,8145.726800,56.590967,160.983805,0.993288
3194,-0.073151,0.106292,0.001943,0.592345,60.053164,0.209515,0.286325,-0.113551,0.337298,6.086529,65.143511,70.598473,8181.719064,54.088788,-807.051418,0.962528
3195,-0.048778,0.127130,0.001894,0.722706,62.834283,0.299958,0.288942,-0.083747,0.330780,5.910030,79.221374,68.430534,8216.991483,54.281485,335.300537,0.937167


In [20]:
def objective(trial, df):
    window_slow = trial.suggest_int("window_slow", 8, 52)
    window = trial.suggest_int("window", 6, window_slow-1)
    window_fast = trial.suggest_int("window_fast", 3, window-1)
    # x_subsample = trial.suggest_float("x_subsample", 0.2, 1)
    
    X, y = create_dataset(df,
                          lag=trial.suggest_int("lag", 0, 5), 
                          window=window, window_slow=window_slow, window_fast=window_fast
    )
    
    model = RandomForestClassifier(
        n_estimators=25, #trial.suggest_int("n_estimators", 10, 250, step=5),
        criterion='entropy', # trial.suggest_categorical("criterion", ['gini', 'entropy']),
        max_depth=trial.suggest_int("max_depth", 1, 8),
        max_features=trial.suggest_int("max_features", int(np.sqrt(len(X.columns))/2), len(X.columns)),
        #max_samples=trial.suggest_float("max_samples", 0.2, 1, step=0.05),
        min_samples_split=trial.suggest_int("min_samples_split", 2, 2**6, step=2),
        #max_leaf_nodes=trial.suggest_int("max_leaf_nodes", 2, 64),
        # class_weight='balanced',
        random_state=42,
        # n_jobs=-1
    )
    
    n_splits = 3
    auc = 0
    acc = 0
    cv = sklearn.model_selection.TimeSeriesSplit(n_splits=n_splits, gap=7, test_size=7*5)
    for train_index, test_index in cv.split(X):
        X_train, X_test = X.iloc[train_index], X.iloc[test_index]
        y_train, y_test = y.iloc[train_index], y.iloc[test_index]
        # print (f"Train size: {train_index.size}; Test size: {test_index.size}")
        # X_train = X_train.sample(int(len(X_train) * x_subsample))
        # y_train = y_train[X_train.index]

        model.fit(X_train, y_train)
        y_p = model.predict_proba(X_test)[:, 1]
        auc += sklearn.metrics.roc_auc_score(y_test, y_p)
        acc += sklearn.metrics.accuracy_score(y_test, y_p >= 0.5)
    
    trial.set_user_attr("auc", auc / n_splits)
    trial.set_user_attr("acc", acc / n_splits)

    #return (acc / n_splits) + (auc / n_splits)*0.25
    return auc / n_splits


study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(multivariate=True, seed=42))
study.optimize(lambda t: objective(t, df), n_trials=100)
print(study.best_params)

/Users/anzekravanja/Projects/smart-trader/conda-env/lib/python3.8/site-packages/optuna/samplers/_tpe/sampler.py:282: ExperimentalWarning: ``multivariate`` option is an experimental feature. The interface can change in the future.
  warnings.warn(


{'window_slow': 49, 'window': 45, 'window_fast': 41, 'lag': 0, 'max_depth': 2, 'max_features': 6, 'min_samples_split': 56}


In [21]:
print(study.best_params)
print(study.best_trial.values)
print(study.best_trial.user_attrs)

{'window_slow': 49, 'window': 45, 'window_fast': 41, 'lag': 0, 'max_depth': 2, 'max_features': 6, 'min_samples_split': 56}
[0.6840167661267404]
{'auc': 0.6840167661267404, 'acc': 0.5238095238095238}


In [ ]:
{'window_slow': 43, 'window': 38, 'window_fast': 35, 'lag': 0, 'criterion': 'gini', 'max_depth': 4, 'max_features': 3, 'min_samples_split': 24}
[0.7084105332954439]
{'auc': 0.7084105332954439, 'acc': 0.6190476190476191}